In [27]:
import pandas as pd
import sys
from pathlib import Path
sys.path.append("../src")
from preprocessing import create_preprocessor

FE_DF_PATH = Path("../data/processed/ames_housing_featured_engineered.parquet")
fe_df = pd.read_parquet(FE_DF_PATH)

In [28]:
# look at the df
fe_df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Was Remodelled,Remod Age,Total Bathrooms,Total Porch SF,Overall Qual Gr Liv,Total Rooms Gr Liv,Built Decade,Has Pool,Has Fireplace,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,None,IR1,Lvl,...,0,50,2.0,272,9936,11592,1960,0,1,215000
1,2,526350040,20,RH,80.0,11622,Pave,None,Reg,Lvl,...,0,49,1.0,260,4480,4480,1960,0,0,105000
2,3,526351010,20,RL,81.0,14267,Pave,None,IR1,Lvl,...,0,52,1.5,429,7974,7974,1950,0,0,172000
3,4,526353030,20,RL,93.0,11160,Pave,None,Reg,Lvl,...,0,42,3.5,0,14770,16880,1960,0,1,244000
4,5,527105010,60,RL,74.0,13830,Pave,None,IR1,Lvl,...,1,12,2.5,246,8145,9774,1990,0,1,189900


In [29]:
# get features and target
X = fe_df.drop("SalePrice", axis=1)
y = fe_df.SalePrice

In [30]:
# create training and test set
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [31]:
# select num. and cat. features
num_features = X.select_dtypes(include="number").columns
cat_features = [col for col in X.columns if col not in num_features]

In [32]:
# create preprocessor and pipelines
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

preprocessor = create_preprocessor(num_features, cat_features)

models_pipeline = {
    "Dummy": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", DummyRegressor(strategy="mean"))
    ]),
    
    "SVR": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", SVR(kernel="rbf"))
    ]),
    "Random Forest Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))
    ]),
    "XG Boost": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", XGBRegressor(n_estimators=100, random_state=42))
    ]),
}

In [ ]:
# train models and calculate metrics
import joblib
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

results = {}

for model_name, pipeline in models_pipeline.items():
    cv_scores = cross_val_score(pipeline,
                                X_train,
                                y_train,
                                cv=5,
                                scoring="r2",
                                n_jobs=1)
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    results[model_name] = {
        "CV Mean R²": cv_scores.mean(),
        "CV Std": cv_scores.std(),
        "Test R² score": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "MAPE": mean_absolute_percentage_error(y_test, y_pred)
    }

    filename = model_name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipeline, f"../outputs/models/{filename}")

In [34]:
# save the metrics df
metrics_df = pd.DataFrame(results).T.round(3)
metrics_df.to_csv("../data/processed/model_metrics.csv")